# ハンズオン① NumPy で線形回帰・勾配降下を手実装

**対応セクション**: 1-1「関数として AI を捉える」  
**推奨所要時間**: 約 30 分  
**使用ライブラリ**: NumPy, Matplotlib のみ（sklearn 不使用）

---

## このノートブックの目標

1. 線形回帰を「パラメータ付きの関数」として実装する
2. MSE 損失関数を手で計算する
3. 勾配降下法で `w` と `b` を自動調整し、「学習」を体感する
4. 損失曲線と学習率の関係を実験で確認する

> **方針**: sklearn 等のライブラリを使わず、for ループで重みを更新します。  
> 「中身が見える」実装を通して、学習の本質を掴んでください。

## セットアップ

In [ ]:
# 実行時間: 数秒
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# 日本語フォント設定（環境に応じて変更してください）
plt.rcParams['font.family'] = 'IPAexGothic'
plt.rcParams['axes.unicode_minus'] = False

# 再現性のためシードを固定
np.random.seed(42)

print('セットアップ完了')

---

## Step 1: データの生成

「部屋の広さ（m²）から家賃（万円）を予測する」問題を想定します。  
真のモデルを `y = 2x + 1` として、そこにノイズを加えた観測値を生成します。

$$
y_i = 2x_i + 1 + \epsilon_i \quad (\epsilon_i \sim \mathcal{N}(0, 2^2))
$$

> **ポイント**: 現実のデータには必ずノイズが含まれます。  
> 学習後のパラメータが完全に `w=2, b=1` にならなくても正常です。

In [ ]:
# 実行時間: 数秒

# ── データ生成 ──────────────────────────────────────
N = 50                                          # データ点数
x = np.linspace(0, 10, N)                      # 入力（0〜10 の等間隔）
y_true = 2 * x + 1                             # 真のモデル（ノイズなし）
y = y_true + np.random.randn(N) * 2            # 観測値（ノイズあり）

print(f'データ数: {N}')
print(f'x の範囲: {x.min():.1f} 〜 {x.max():.1f}')
print(f'y の範囲: {y.min():.2f} 〜 {y.max():.2f}')

# ── 散布図 ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, color='#57534E', s=25, alpha=0.7, label='観測値（ノイズあり）')
ax.plot(x, y_true, color='#0D4A38', linewidth=1.5, linestyle='--', label='真のモデル (y = 2x + 1)')
ax.set_xlabel('x（部屋の広さ）')
ax.set_ylabel('y（家賃）')
ax.set_title('生成データの散布図')
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 2: モデルの定義

線形回帰モデル $\hat{y} = wx + b$ をそのまま関数として実装します。  
パラメータ `w`（重み）と `b`（バイアス）は、最初はゼロで初期化します。

In [ ]:
# 実行時間: 数秒

def predict(x, w, b):
    """線形回帰モデル: ŷ = wx + b"""
    return w * x + b

# 初期パラメータ（何も学習していない状態）
w, b = 0.0, 0.0

y_pred_init = predict(x, w, b)
print(f'初期パラメータ: w={w}, b={b}')
print(f'初期予測値の例: {y_pred_init[:5]}')  # すべて 0.0
print('→ まったく予測できていない')

---

## Step 3: 損失関数（MSE）の実装

予測値と正解値の「外れ具合」を数値で表します。

$$
L(w, b) = \frac{1}{N} \sum_{i=1}^{N} \left( y_i - \hat{y}_i \right)^2
$$

In [ ]:
# 実行時間: 数秒

def mse_loss(y_true, y_pred):
    """平均二乗誤差（MSE）を計算する"""
    return np.mean((y_true - y_pred) ** 2)

# 初期パラメータでの損失
loss_init = mse_loss(y, predict(x, w=0.0, b=0.0))
print(f'初期損失 (w=0, b=0):    {loss_init:.4f}')

# 真のパラメータでの損失（ノイズの分だけ残る）
loss_true = mse_loss(y, predict(x, w=2.0, b=1.0))
print(f'真のパラメータの損失:   {loss_true:.4f}')
print(f'\n損失の比: {loss_init / loss_true:.1f} 倍の差がある')

---

## Step 4: 勾配降下法の実装

損失 $L$ を `w` と `b` それぞれで偏微分した「勾配」を計算し、  
その反対方向に学習率 $\alpha$ だけパラメータを動かします。

$$
\frac{\partial L}{\partial w} = -\frac{2}{N} \sum_i x_i(y_i - \hat{y}_i), \quad
\frac{\partial L}{\partial b} = -\frac{2}{N} \sum_i (y_i - \hat{y}_i)
$$

$$
w \leftarrow w - \alpha \frac{\partial L}{\partial w}, \quad b \leftarrow b - \alpha \frac{\partial L}{\partial b}
$$

In [ ]:
# 実行時間: 約5秒

# ── ハイパーパラメータ ──────────────────────────────
alpha    = 0.01   # 学習率
n_epochs = 300    # 更新回数

# ── 初期化 ──────────────────────────────────────────
w, b = 0.0, 0.0
history = []      # エポックごとの損失を記録

# ── 勾配降下ループ ────────────────────────────────────
for epoch in range(n_epochs):

    # 1. 予測
    y_pred = predict(x, w, b)

    # 2. 損失の計算
    loss = mse_loss(y, y_pred)
    history.append(loss)

    # 3. 勾配の計算（MSE の偏微分）
    dw = -2 / N * np.sum(x * (y - y_pred))
    db = -2 / N * np.sum(y - y_pred)

    # 4. パラメータの更新
    w = w - alpha * dw
    b = b - alpha * db

    if epoch % 50 == 0:
        print(f'Epoch {epoch:3d} | loss={loss:.4f} | w={w:.4f} | b={b:.4f}')

print(f'\n--- 最終結果 ---')
print(f'w = {w:.4f}  （真の値: 2.0）')
print(f'b = {b:.4f}  （真の値: 1.0）')

---

## Step 5: 損失曲線の可視化

エポックが進むにつれて損失が下がっていく様子を確認します。  
これが「学習が進んでいる」ことの証拠です。

In [ ]:
# 実行時間: 数秒

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(history, color='#0D4A38', linewidth=1.8)
ax.axhline(y=loss_true, color='#A8A29E', linestyle='--', linewidth=1, label=f'理論最小値（ノイズ分）= {loss_true:.2f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('損失曲線（学習の進捗）')
ax.legend()
plt.tight_layout()
plt.show()

print(f'初期損失: {history[0]:.2f} → 最終損失: {history[-1]:.2f}')

---

## Step 6: 学習後の予測結果を可視化

In [ ]:
# 実行時間: 数秒

y_pred_final = predict(x, w, b)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 左: 学習前
axes[0].scatter(x, y, color='#57534E', s=25, alpha=0.6)
axes[0].plot(x, predict(x, 0.0, 0.0), color='#EF4444', linewidth=2, label='初期 (w=0, b=0)')
axes[0].set_title('学習前')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].legend()

# 右: 学習後
axes[1].scatter(x, y, color='#57534E', s=25, alpha=0.6)
axes[1].plot(x, y_pred_final, color='#0D4A38', linewidth=2, label=f'学習後 (w={w:.3f}, b={b:.3f})')
axes[1].plot(x, y_true, color='#A8A29E', linewidth=1.5, linestyle='--', label='真のモデル (w=2, b=1)')
axes[1].set_title('学習後')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
axes[1].legend()

plt.tight_layout()
plt.show()

---

## 実験: 学習率（alpha）を変えてみよう

学習率が損失曲線の形にどう影響するかを比較します。

| alpha | 予想される挙動 |
|---|---|
| `0.001` | 収束が遅いが安定 |
| `0.01` | バランスが良い |
| `0.05` | 速いが振動が出ることも |
| `0.2` | 発散する可能性が高い |

In [ ]:
# 実行時間: 約10秒

alphas = [0.001, 0.01, 0.05, 0.2]
colors = ['#0D4A38', '#1A6B52', '#7C5C00', '#991B1B']

fig, ax = plt.subplots(figsize=(8, 4))

for alpha_exp, color in zip(alphas, colors):
    w_exp, b_exp = 0.0, 0.0
    hist_exp = []

    for epoch in range(300):
        y_pred_exp = predict(x, w_exp, b_exp)
        loss_exp = mse_loss(y, y_pred_exp)
        hist_exp.append(min(loss_exp, 500))  # 発散時の表示制限

        dw = -2 / N * np.sum(x * (y - y_pred_exp))
        db = -2 / N * np.sum(y - y_pred_exp)
        w_exp -= alpha_exp * dw
        b_exp -= alpha_exp * db

    ax.plot(hist_exp, color=color, linewidth=1.8, label=f'alpha={alpha_exp}')

ax.axhline(y=loss_true, color='#A8A29E', linestyle='--', linewidth=1, label='理論最小値')
ax.set_ylim(-5, 200)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('学習率による損失曲線の違い')
ax.legend()
plt.tight_layout()
plt.show()

---

## まとめ

このノートブックで体感したこと:

1. **線形回帰** は $\hat{y} = wx + b$ という「パラメータ付きの関数」
2. **MSE 損失** は「予測の外れ具合」を1つの数値にした指標
3. **勾配降下法** は「損失が下がる方向にパラメータを少しずつ動かす」繰り返し  
   → これが「**学習**」の正体
4. **学習率** が大きすぎると発散、小さすぎると収束が遅い

次のハンズオンでは、この流れを PyTorch で自動化し、  
2層ニューラルネットワークによる MNIST 分類に挑戦します。